## Initial Exploration and Familiarisation

This includes the data downloading and then general familiarisation with the data and figuring out how the dataset looks and how to approach the analysis.

In [2]:
import numpy as np
import pandas as pd

# using kaggle for dataset as sklifts dataset no longer works (aws link is broken)

import kagglehub
path = kagglehub.dataset_download("arashnic/uplift-modeling")

print("Path to dataset files:", path)

Resuming download from 29360128 bytes (310796575 bytes left)...
Resuming download to /home/sid/.cache/kagglehub/datasets/arashnic/uplift-modeling/1.archive (29360128/340156703) bytes left.


100%|██████████| 324M/324M [01:48<00:00, 2.86MB/s] 

Extracting files...


Path to dataset files: /home/sid/.cache/kagglehub/datasets/arashnic/uplift-modeling/versions/1


In [8]:
df_sample = pd.read_csv('/home/sid/.cache/kagglehub/datasets/arashnic/uplift-modeling/versions/1/criteo-uplift-v2.1.csv', nrows=10)

print(df_sample.shape)
print(df_sample.dtypes)
df_sample

(10, 16)
f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
5,12.616365,10.059654,8.904507,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
6,12.616365,10.059654,8.783340,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
7,12.616365,10.059654,8.964528,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
8,12.616365,10.059654,9.037809,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
9,12.616365,10.059654,9.015128,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


### Data Auditing

Before any actual modelling we test the data for completeness, consistenct, labels, etc.

In [18]:
df = pd.read_csv('/home/sid/.cache/kagglehub/datasets/arashnic/uplift-modeling/versions/1/criteo-uplift-v2.1.csv')

FEATURE_COLS = [f"f{i}" for i in range(12)]
LABEL_COLS = ["treatment", "conversion", "visit", "exposure"]

missing = df.isna().sum()
print(f"Missing values per column: {missing}")

n_dup = df.duplicated().sum()
n_dup_feat = df.duplicated(subset=FEATURE_COLS).sum()

print(f"\nExact duplicate rows (all 16 columns): {n_dup:,} ({n_dup / len(df):.2%})")
print(f"Rows with duplicated features only:    {n_dup_feat:,} ({n_dup_feat / len(df):.2%})")


Missing values per column: f0            0
f1            0
f2            0
f3            0
f4            0
f5            0
f6            0
f7            0
f8            0
f9            0
f10           0
f11           0
treatment     0
conversion    0
visit         0
exposure      0
dtype: int64

Exact duplicate rows (all 16 columns): 1,259,545 (9.01%)
Rows with duplicated features only:    1,626,259 (11.63%)


In [19]:
def group_summary(data):
    return pd.Series({
        "users": len(data),
        "visit_rate": data["visit"].mean(),
        "conversion_rate": data["conversion"].mean(),
        "exposure_rate": data["exposure"].mean(),
    })

summary = df.groupby("treatment").apply(group_summary)
summary.index = summary.index.map({0: "control", 1: "treatment"})
summary.loc["overall"] = group_summary(df)
summary["share"] = summary["users"] / len(df)

summary.style.format({
    "users": "{:,.0f}",
    "share": "{:.2%}",
    "visit_rate": "{:.3%}",
    "conversion_rate": "{:.4%}",
    "exposure_rate": "{:.3%}",
})

,users,visit_rate,conversion_rate,exposure_rate,share
treatment,,,,,
control,"2,096,937",3.820%,0.1938%,0.000%,15.00%
treatment,"11,882,655",4.854%,0.3089%,3.604%,85.00%
overall,"13,979,592",4.699%,0.2917%,3.063%,100.00%


In [ ]:
"""
logical consistency
Expected relationships: control users should never be exposed to the ad, and a conversion should normally follow a visit.
"""
print("Treatment x exposure:")
print(pd.crosstab(df["treatment"], df["exposure"], margins=True), "\n")

print("Visit x conversion:")
print(pd.crosstab(df["visit"], df["conversion"], margins=True), "\n")

checks = {
    "Control users with exposure = 1": ((df["treatment"] == 0) & (df["exposure"] == 1)).sum(),
    "Conversions without a visit": ((df["conversion"] == 1) & (df["visit"] == 0)).sum(),
}
for name, n in checks.items():
    print(f"{name:<34} {n:,}")

treated = df[df["treatment"] == 1]
print(f"\nShare of treated users actually exposed: {treated['exposure'].mean():.2%}")

Treatment x exposure:
exposure          0       1       All
treatment                            
0           2096937       0   2096937
1          11454443  428212  11882655
All        13551380  428212  13979592 

Visit x conversion:
conversion         0      1       All
visit                                
0           13322663      0  13322663
1             616155  40774    656929
All         13938818  40774  13979592 

Control users with exposure = 1    0
Conversions without a visit        0

Share of treated users actually exposed: 3.60%


If assignment is random, feature distributions should be nearly identical across groups. With 14 million rows, even tiny differences are statistically significant, so balance is judged by the standardized mean difference (SMD) rather than p-values. An absolute SMD below 0.1 is a common threshold for good balance.

In [ ]:
is_treated = df["treatment"] == 1
rows = []
for c in FEATURE_COLS:
    x = df[c].astype("float64")
    m1, m0 = x[is_treated].mean(), x[~is_treated].mean()
    v1, v0 = x[is_treated].var(), x[~is_treated].var()
    rows.append({
        "feature": c,
        "mean_treatment": m1,
        "mean_control": m0,
        "smd": (m1 - m0) / np.sqrt((v1 + v0) / 2), #smd discussed more in background_notes notebok
    })

balance = pd.DataFrame(rows).set_index("feature")
balance["abs_smd"] = balance["smd"].abs()
print(f"Max |SMD|: {balance['abs_smd'].max():.4f} ({balance['abs_smd'].idxmax()})")
print(f"Features with |SMD| >= 0.1: {(balance['abs_smd'] >= 0.1).sum()}")
balance.sort_values("abs_smd", ascending=False)

Max |SMD|: 0.0488 (f3)
Features with |SMD| >= 0.1: 0


,mean_treatment,mean_control,smd,abs_smd
feature,,,,
f3,4.169412,4.232821,-0.048836,0.048836
f6,-4.182792,-3.999880,-0.040448,0.040448
f5,4.026602,4.039339,-0.030557,0.030557
f9,16.052589,15.886253,0.024001,0.024001
f1,10.070337,10.067935,0.023994,0.023994
f8,3.933392,3.934652,-0.022433,0.022433
f7,5.105555,5.080284,0.021270,0.021270
f10,5.333660,5.331898,0.010563,0.010563
f4,10.339245,10.336526,0.007966,0.007966


Also discussed in background notes notebook  but here we train a classifier to predict treatment from the 12 features. Under true randomisation, it should do no better than chance (ROC-AUC close to 0.5). Using just 1 million-row random sample 

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

SEED = 42

sample = df.sample(n=1_000_000, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    sample[FEATURE_COLS], sample["treatment"],
    test_size=0.3, stratify=sample["treatment"], random_state=SEED,
)

clf = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.1, num_leaves=31,
    random_state=SEED, verbose=-1,
)
clf.fit(X_tr, y_tr)
auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
print(f"ROC-AUC for predicting treatment from features: {auc:.4f}")

ROC-AUC for predicting treatment from features: 0.5124


To judge whether an AUC of about 0.51 is meaningful, the same classifier is trained on randomly shuffled treatment labels. The spread of these AUCs shows what pure chance looks like at this sample size.


In [26]:
rng = np.random.default_rng(SEED)
perm_aucs = []
for i in range(3):
    y_tr_perm = rng.permutation(y_tr.values)
    y_te_perm = rng.permutation(y_te.values)
    clf_perm = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.1, num_leaves=31,
        random_state=SEED, verbose=-1,
    )
    clf_perm.fit(X_tr, y_tr_perm)
    perm_aucs.append(roc_auc_score(y_te_perm, clf_perm.predict_proba(X_te)[:, 1]))

print(f"Real-label AUC:      {auc:.4f}")
print("Shuffled-label AUCs:", [round(float(a), 4) for a in perm_aucs])

importance = pd.Series(
    clf.booster_.feature_importance(importance_type="gain"), index=FEATURE_COLS
)
print("\nFeatures that best separate treatment from control (gain share):")
print((importance / importance.sum()).sort_values(ascending=False).round(3))

Real-label AUC:      0.5124
Shuffled-label AUCs: [0.4976, 0.4996, 0.4999]

Features that best separate treatment from control (gain share):
f0     0.169
f8     0.153
f2     0.151
f6     0.150
f9     0.105
f7     0.075
f10    0.072
f3     0.062
f4     0.031
f5     0.015
f11    0.011
f1     0.007
dtype: float64
